# Orthogonal-head follow-ups

Clones `kaggle/controlled-ortho` and runs identity-proj, n_embd=32, and lambda=1.0 jobs from the repo.

In [ ]:
import os, sys, platform, subprocess, json, shutil
from pathlib import Path

print("python", sys.version)
print("platform", platform.platform())
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))


In [ ]:
REPO = "https://github.com/Priyanshu-5257/failed_hypothesis.git"
BRANCH = "kaggle/controlled-ortho"
REPO_DIR = Path("/kaggle/working/repo")
WORK = Path("/kaggle/working/followup")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, str(REPO_DIR)])
subprocess.check_call(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"])


In [ ]:
%pip -q install pytest


In [ ]:
def run(cmd, cwd=None):
    print("+", *cmd, flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(rc)

run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=str(REPO_DIR))
WORK.mkdir(parents=True, exist_ok=True)
run(
    [
        sys.executable, "-m", "ortho_attn.sweep",
        "--data", str(REPO_DIR / "input.txt"),
        "--work", str(WORK),
        "--max-iters", "5000",
        "--eval-iters", "200",
    ],
    cwd=str(REPO_DIR),
)
report = json.loads((WORK / "comparison.json").read_text())
print(json.dumps(report, indent=2))
(Path("/kaggle/working") / "ok.json").write_text(
    json.dumps({"ok": True, "branch": BRANCH, "report": report}, indent=2)
)
